In [69]:
import pandas as pd

In [70]:
df = pd.read_csv('synthetic_dataset.csv')

In [71]:
df

,Category,Price,Rating,Stock,Discount
0,NaN,5548.0,1.870322,NaN,0.0
1,NaN,3045.0,4.757798,NaN,38.0
2,NaN,4004.0,NaN,In Stock,0.0
3,NaN,4808.0,1.492085,NaN,33.0
4,NaN,1817.0,NaN,Out of Stock,23.0
...,...,...,...,...,...
4357,NaN,4436.0,4.728335,NaN,49.0
4358,B,6236.0,NaN,Out of Stock,4.0
4359,NaN,3283.0,NaN,Out of Stock,9.0
4360,D,2999.0,4.425995,NaN,40.0


In [84]:
print(f"Размер данных: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")
print("\nПропуски (%):")
print(df.isnull().sum()/len(df)*100)

Размер данных: (4362, 5)
Колонки: ['Category', 'Price', 'Rating', 'Stock', 'Discount']

Пропуски (%):
Category    62.998624
Price        3.988996
Rating      46.996790
Stock       30.994956
Discount     8.986703
dtype: float64


In [98]:
print("Уникальные значения Category")
print(df['Category'].unique())

Уникальные значения Category
[nan 'C' 'A' 'B' 'D']


In [105]:
with_cat = df[df['Category'].notna()]
without_cat = df[df['Category'].isna()]
print(f"Товары с категорией: {len(with_cat)}")
print(f"Товары без категории: {len(without_cat)}")

Товары с категорией: 1614
Товары без категории: 2748


In [107]:
print(f"Средняя цена с категорией: {with_cat['Price'].mean():.2f}")
print(f"Средняя цена без категории: {without_cat['Price'].mean():.2f}")

Средняя цена с категорией: 5073.63
Средняя цена без категории: 4983.44


In [108]:
df['Category'] = df['Category'].fillna('Unknown')

✓ Category: пропуски заполнены 'Unknown'


In [109]:
print(f"Строк без цены: {df['Price'].isnull().sum()}")
print(f"Это {df['Price'].isnull().mean()*100:.1f}% данных")

Строк без цены: 174
Это 4.0% данных


In [112]:
rows_before = len(df)
df = df.dropna(subset=['Price'])
rows_after = len(df)
print(f"Price: удалено {rows_before - rows_after} строк ({(rows_before - rows_after) / rows_before * 100:.1f}%)")

✓ Price: удалено 174 строк (4.0%)


In [113]:
has_rating = df[df['Rating'].notna()]
no_rating = df[df['Rating'].isna()]

print("Сравнение цен:")
print(f"С рейтингом: {has_rating['Price'].mean():.2f}")
print(f"Без рейтинга: {no_rating['Price'].mean():.2f}")

Сравнение цен:
С рейтингом: 5042.64
Без рейтинга: 4987.88


In [115]:
df = df.copy()
df.loc[:, 'Rating_Missing'] = df['Rating'].isnull().astype(int)
df.loc[:, 'Rating'] = df.groupby('Category')['Rating'].transform(
    lambda x: x.fillna(x.median())
)

In [116]:
print("Распределение Stock:")
print(df['Stock'].value_counts(dropna=False))

print("\nСредняя цена по статусу:")
print(df.groupby(df['Stock'].fillna('Missing'))['Price'].mean())

Распределение Stock:
Stock
In Stock        1454
Out of Stock    1426
NaN             1308
Name: count, dtype: int64

Средняя цена по статусу:
Stock
In Stock        5042.933975
Missing         4999.940367
Out of Stock    5006.118513
Name: Price, dtype: float64


In [117]:
df['Stock_Missing'] = df['Stock'].isnull().astype(int)

df['Stock'] = df['Stock'].fillna('Unknown')

✓ Stock: создан флаг, пропуски заполнены 'Unknown'


In [118]:
print("Статистика скидок:")
print(df['Discount'].describe())

print("\nСколько товаров со скидкой 0:")
zero_count = (df['Discount'] == 0).sum()
print(f"Скидка 0: {zero_count} ({zero_count/len(df)*100:.1f}%)")

Статистика скидок:
count    3809.000000
mean       24.467314
std        14.316777
min         0.000000
25%        12.000000
50%        24.000000
75%        37.000000
max        49.000000
Name: Discount, dtype: float64

Сколько товаров со скидкой 0:
Скидка 0: 75 (1.8%)


In [119]:
df['Discount_Missing'] = df['Discount'].isnull().astype(int)

df['Discount'] = df['Discount'].fillna(0)


✓ Discount: создан флаг, пропуски заполнены 0


In [120]:
# Проверка
print("\nПОСЛЕ ОЧИСТКИ:")
print(df.isnull().sum())

# Сохраняем
df.to_csv('cleaned_data.csv', index=False)
print("\n✓ Данные сохранены в 'cleaned_data.csv'")


ПОСЛЕ ОЧИСТКИ:
Category            0
Price               0
Rating              0
Stock               0
Discount            0
Rating_Missing      0
Stock_Missing       0
Discount_Missing    0
dtype: int64

✓ Данные сохранены в 'cleaned_data.csv'


In [121]:
# 1. Распределение по категориям
print(df['Category'].value_counts())

# 2. Средняя цена по категориям
print(df.groupby('Category')['Price'].mean())

# 3. Средний рейтинг по наличию скидки
print(df.groupby(df['Discount'] > 0)['Rating'].mean())

# 4. Корреляция между ценой и рейтингом
print(df[['Price', 'Rating']].corr())

Category
Unknown    2631
C           400
D           399
A           389
B           369
Name: count, dtype: int64
Category
A          4782.910026
B          5410.330623
C          5086.802500
D          5032.468672
Unknown    4983.440897
Name: Price, dtype: float64
Discount
False    2.991547
True     3.068465
Name: Rating, dtype: float64
          Price   Rating
Price   1.00000  0.01626
Rating  0.01626  1.00000


✅ Все пропуски обработаны
✅ Созданы полезные флаги
✅ Данные готовы к анализу
✅ Есть четкая документация каждого решения